# Entrenamiento y generación del plan nutricional

**CC3045 – Inteligencia Artificial | Smart Diet Planner | Parte 2**

Este notebook demuestra el pipeline completo de Parte 2: generación del plan semanal personalizado y evaluación de su calidad nutricional.

| Paso | Módulo | Técnica |
|------|--------|---------|
| Perfil del usuario | `modules/profile.py` | Fórmula Mifflin-St Jeor + TDEE |
| Filtrado de alimentos | `modules/filter.py` | Árbol de Decisión |
| Recomendación | `modules/recommender.py` | K-Nearest Neighbors |
| Plan semanal | `modules/planner.py` | Agente Greedy |
| Evaluación | `modules/evaluator.py` | Métricas nutricionales |

## B. Importación de librerías y módulos

In [ ]:
import sys
import os

# Configurar encoding UTF-8 para compatibilidad en Windows
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# Agregar el directorio raíz del proyecto al path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)

# Módulos del proyecto
from modules.profile     import calcular_perfil
from modules.filter      import filtrar_alimentos
from modules.recommender import get_recomendaciones
from modules.planner     import generar_plan_semanal, plan_a_dataframe, lista_compras
from modules.evaluator   import evaluar_plan, imprimir_reporte

print('✅ Módulos cargados correctamente.')

## C. Carga del dataset

In [ ]:
DATA_PATH = os.path.join(ROOT, 'data', 'processed', 'foods_clean.csv')
df = pd.read_csv(DATA_PATH)

print(f'📦 Dataset cargado: {len(df):,} alimentos')
print(f'   Columnas: {list(df.columns)}')
print(f'   Rango de calorías: {df["calories"].min():.0f} – {df["calories"].max():.0f} kcal/100g')
print()
df.head(5)

## D. Creación del perfil de usuario

In [ ]:
# Perfil de usuario de prueba
perfil = calcular_perfil(
    peso_kg=70,
    altura_cm=175,
    edad=25,
    sexo='masculino',
    nivel_actividad='moderado',
    meta='perder peso'
)

print('👤 PERFIL NUTRICIONAL GENERADO')
print('=' * 40)
for clave, valor in perfil.items():
    unidad = 'kcal/día' if 'calor' in clave or clave in ('bmr', 'tdee') else 'g/día' if '_g' in clave else ''
    print(f'  {clave:22}: {valor} {unidad}')

## E. Filtrado de alimentos

Aplica el pipeline de filtrado:
1. **Filtrado por reglas**: elimina alimentos con restricciones (gluten, lactosa), exceso de calorías, grasa o azúcar
2. **Árbol de Decisión**: clasifica alimentos como "aptos" o "no aptos" para el perfil

In [ ]:
restricciones = ['gluten', 'lactosa']

print(f'📋 Alimentos antes del filtrado: {len(df):,}')
print(f'🚫 Restricciones activas: {", ".join(restricciones)}')
print()

df_aptos, modelo_dt, accuracy_dt = filtrar_alimentos(
    df,
    perfil,
    restricciones=restricciones,
    max_cal=700,
    max_grasa=40,
    max_azucar=30
)

print()
print(f'✅ Alimentos aptos finales: {len(df_aptos):,} ({len(df_aptos)/len(df)*100:.1f}% del total)')
print(f'🌳 Accuracy del árbol de decisión: {accuracy_dt:.2%}')

In [ ]:
# Visualización: distribución calórica antes vs después del filtrado
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['calories'].clip(upper=800), bins=50, color='#e74c3c', alpha=0.7, edgecolor='white')
axes[0].set_title('Distribución calórica — Dataset completo', fontsize=12)
axes[0].set_xlabel('Calorías por 100g')
axes[0].set_ylabel('Cantidad de alimentos')

axes[1].hist(df_aptos['calories'].clip(upper=800), bins=50, color='#27ae60', alpha=0.7, edgecolor='white')
axes[1].set_title('Distribución calórica — Alimentos aptos', fontsize=12)
axes[1].set_xlabel('Calorías por 100g')

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'data', 'processed', 'distribucion_filtrado.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Gráfica guardada en data/processed/distribucion_filtrado.png')

## F. Recomendación personalizada de alimentos

El modelo KNN encuentra alimentos nutricionalmente similares a las preferencias del usuario dentro del conjunto de alimentos aptos.

In [ ]:
preferencias = ['chicken', 'salmon']

df_recomendados, modelo_knn, scaler = get_recomendaciones(
    df_aptos,
    perfil,
    preferencias=preferencias,
    k=8,
    n_total=80
)

print()
print(f'✅ Pool de recomendaciones generado: {len(df_recomendados):,} alimentos')

In [ ]:
# Mostrar top 15 recomendaciones
cols_mostrar = ['nombre', 'calories', 'protein', 'carbs', 'fat', 'fiber']
display_cols = [c for c in cols_mostrar if c in df_recomendados.columns]
print('Top 15 alimentos recomendados:')
df_recomendados[display_cols].head(15)

## G. Generación del plan semanal

El agente greedy construye el plan de 7 días. Cada día tiene 4 comidas con distribución calórica:
- **Desayuno** 25% · **Almuerzo** 35% · **Cena** 30% · **Snack** 10%

In [ ]:
plan = generar_plan_semanal(df_recomendados, perfil, dias=7)
plan_df = plan_a_dataframe(plan)

print()
print(f'✅ Plan generado: {len(plan_df)} entradas ({len(plan_df)//4} días × 4 comidas)')

In [ ]:
# Vista tabular del plan completo
print('Plan semanal generado:')
plan_df

In [ ]:
# Visualización: calorías por día vs objetivo
cal_por_dia = plan_df.groupby('dia')['calories'].sum()
dias_orden = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
cal_por_dia = cal_por_dia.reindex([d for d in dias_orden if d in cal_por_dia.index])

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(cal_por_dia.index, cal_por_dia.values, color='#3498db', alpha=0.8, edgecolor='white')
ax.axhline(perfil['calorias_objetivo'], color='#e74c3c', linewidth=2,
           linestyle='--', label=f'Objetivo: {perfil["calorias_objetivo"]:.0f} kcal')
ax.axhline(perfil['calorias_objetivo'] * 1.05, color='#f39c12', linewidth=1,
           linestyle=':', alpha=0.7, label='±5%')
ax.axhline(perfil['calorias_objetivo'] * 0.95, color='#f39c12', linewidth=1,
           linestyle=':', alpha=0.7)

for bar, val in zip(bars, cal_por_dia.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)

ax.set_title('Calorías diarias del plan vs. objetivo', fontsize=13)
ax.set_ylabel('Calorías (kcal)')
ax.legend()
ax.set_ylim(0, max(cal_por_dia.values) * 1.15)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'data', 'processed', 'calorias_plan_semanal.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Gráfica guardada en data/processed/calorias_plan_semanal.png')

## H. Lista de compras

In [ ]:
compras_df = lista_compras(plan_df)

print(f'🛒 Lista de compras consolidada: {len(compras_df)} alimentos únicos')
print()
compras_df

## I. Evaluación del plan nutricional

Calcula métricas objetivas de calidad:
- **Desviación calórica**: qué tan cerca está cada día del objetivo
- **Cobertura de macros**: porcentaje de días que cumplen proteína, carbos y grasa
- **Diversidad**: ratio de alimentos únicos sobre apariciones totales
- **Precisión de filtrado**: verifica que el plan no incluya alimentos con restricciones
- **Score general**: puntuación 0-100 ponderada de las cuatro métricas

In [ ]:
metricas = evaluar_plan(plan_df, perfil, restricciones=restricciones)
imprimir_reporte(metricas)

In [ ]:
# Detalle de desviación calórica por día
desv = metricas['desviacion_calorica']
print(f'Desviación calórica promedio: {desv["promedio_pct"]}%  {desv["estado"]}')
print(f'Días dentro del  5%: {desv["dias_dentro_5pct"]}/{desv["total_dias"]}')
print(f'Días dentro del 10%: {desv["dias_dentro_10pct"]}/{desv["total_dias"]}')
print()

# Tabla de desviación por día
filas_desv = []
for dia, v in desv['por_dia'].items():
    filas_desv.append({
        'Día': dia,
        'Calorías reales': v['calorias_reales'],
        'Objetivo': v['calorias_objetivo'],
        'Desviación %': v['desviacion_pct'],
        'Cumple (≤10%)': '✅' if v['cumple'] else '❌'
    })
pd.DataFrame(filas_desv)

In [ ]:
# Tabla de cobertura de macronutrientes
cob = metricas['cobertura_macros']
filas_mac = []
for macro in ['protein', 'carbs', 'fat']:
    m = cob[macro]
    filas_mac.append({
        'Macronutriente': macro,
        'Objetivo (g)': m['objetivo_g'],
        'Promedio real (g)': m['promedio_real'],
        'Días OK': m['dias_cumple'],
        'Cobertura %': m['cobertura_pct'],
        'Estado': m['estado']
    })
print(f'Cobertura de macros promedio: {cob["promedio_general_pct"]}%')
pd.DataFrame(filas_mac)

In [ ]:
# Visualización: scores por categoría
s = metricas['score']
categorias = ['Calórico', 'Macros', 'Diversidad', 'Precisión']
valores    = [s['score_calorico'], s['score_macros'], s['score_diversidad'], s['score_precision']]
colores    = ['#3498db', '#e67e22', '#27ae60', '#9b59b6']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(categorias, valores, color=colores, alpha=0.85, edgecolor='white')
ax.axhline(100, color='gray', linewidth=0.8, linestyle='--', alpha=0.4)
ax.set_ylim(0, 115)
ax.set_ylabel('Score')
ax.set_title(f'Score nutricional por categoría  |  Score general: {s["score_final"]}/100  {s["calificacion"]}',
             fontsize=11)

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'data', 'processed', 'score_nutricional.png'), dpi=100, bbox_inches='tight')
plt.show()
print('Gráfica guardada en data/processed/score_nutricional.png')

## J. Guardado de resultados

In [ ]:
PROCESSED = os.path.join(ROOT, 'data', 'processed')

# Guardar plan semanal
plan_path = os.path.join(PROCESSED, 'plan_semanal.csv')
plan_df.to_csv(plan_path, index=False)
print(f'✅ Plan semanal guardado:  {plan_path}')
print(f'   Filas: {len(plan_df)} | Columnas: {list(plan_df.columns)}')

# Guardar lista de compras
compras_path = os.path.join(PROCESSED, 'lista_compras.csv')
compras_df.to_csv(compras_path, index=False)
print(f'\n✅ Lista de compras guardada: {compras_path}')
print(f'   Alimentos únicos: {len(compras_df)}')

# Resumen final
print()
print('=' * 50)
print('RESUMEN FINAL')
print('=' * 50)
print(f'  Usuario        : {perfil["meta"]} | {perfil["nivel_actividad"]}')
print(f'  Objetivo diario: {perfil["calorias_objetivo"]} kcal')
print(f'  Alimentos aptos: {len(df_aptos):,} de {len(df):,}')
print(f'  Recomendados   : {len(df_recomendados)}')
print(f'  Score del plan : {metricas["score"]["score_final"]}/100  {metricas["score"]["calificacion"]}')
print(f'  Restricciones  : {", ".join(restricciones)}')
print(f'  Violaciones    : {len(metricas["precision_filtrado"]["violaciones"])}')
print('=' * 50)